# Phase 8 — Facial Emotion Recognition (EfficientNet-B0)

**Model:** EfficientNet-B0 fine-tuned on FER2013
**Dataset:** FER2013 from Kaggle (35,887 grayscale 48x48 faces, 7 classes)
**Classes:** Angry, Disgust, Fear, Happy, Sad, Surprise, Neutral
**Target:** Validation accuracy >= 0.60

**Setup:** Runtime -> Change runtime type -> **T4 GPU**

## Cell 1: Install Dependencies & GPU Check

In [ ]:
!pip install -q torch torchvision
!pip install -q scikit-learn tqdm kagglehub

import torch
import numpy as np
import os, time, json
from collections import Counter

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU! Runtime -> Change runtime type -> T4 GPU")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Cell 2: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/voice_pipeline_models/facial_emotion"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Models will be saved to: {SAVE_DIR}")

## Cell 3: Download FER2013 Dataset

Downloads FER2013 from Kaggle via kagglehub.
~35,887 images: 28,709 train + 3,589 val + 3,589 test.

In [ ]:
import kagglehub

DATA_DIR = "/content/fer2013"

if os.path.exists(DATA_DIR) and len(os.listdir(DATA_DIR)) > 0:
    print(f"Dataset already at {DATA_DIR}")
else:
    print("Downloading FER2013 from Kaggle...")
    path = kagglehub.dataset_download("msambare/fer2013")
    print(f"Downloaded to: {path}")
    # kagglehub downloads to a cache dir; create symlink or copy
    if not os.path.exists(DATA_DIR):
        os.symlink(path, DATA_DIR)
    print(f"Linked to {DATA_DIR}")

# Check structure
for split in ['train', 'test']:
    split_dir = os.path.join(DATA_DIR, split)
    if os.path.exists(split_dir):
        classes = sorted(os.listdir(split_dir))
        total = sum(len(os.listdir(os.path.join(split_dir, c)))
                    for c in classes if os.path.isdir(os.path.join(split_dir, c)))
        print(f"  {split}: {total} images, classes: {classes}")

## Cell 4: Dataset & DataLoaders

FER2013 from Kaggle (msambare version) comes as folders:
`train/{class_name}/`, `test/{class_name}/`

Images are 48x48 grayscale PNGs. We resize to 224x224 and convert to 3-channel.

In [ ]:
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split

EMOTION_CLASSES = ["angry", "disgust", "fear", "happy", "sad", "surprise", "neutral"]
DISPLAY_CLASSES = ["Angry", "Disgust", "Fear", "Happy", "Sad", "Surprise", "Neutral"]
NUM_CLASSES = 7
BATCH_SIZE = 64

# Training transforms with augmentation
train_transform = T.Compose([
    T.Grayscale(num_output_channels=3),  # 1ch -> 3ch
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    T.RandomErasing(p=0.2),
])

# Validation/test transforms (no augmentation)
val_transform = T.Compose([
    T.Grayscale(num_output_channels=3),
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Load datasets
train_full = ImageFolder(os.path.join(DATA_DIR, "train"), transform=train_transform)
test_dataset = ImageFolder(os.path.join(DATA_DIR, "test"), transform=val_transform)

# Split train into train + val (90/10)
n_train = int(0.9 * len(train_full))
n_val = len(train_full) - n_train
train_dataset, val_dataset_raw = random_split(
    train_full, [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)

# Val dataset needs val_transform (no augmentation)
# Wrap to override transform
class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        img, label = self.subset[idx]
        # img is already transformed by train_transform; we need raw image
        # Since random_split wraps ImageFolder, we access the underlying dataset
        real_idx = self.subset.indices[idx]
        path, target = self.subset.dataset.samples[real_idx]
        from PIL import Image
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, target

val_dataset = TransformSubset(val_dataset_raw, val_transform)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=True)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")
print(f"Classes: {train_full.classes}")

# Class distribution
from collections import Counter
train_labels = [train_full.targets[i] for i in train_dataset.indices]
dist = Counter(train_labels)
print(f"\nTrain class distribution:")
for i, c in enumerate(DISPLAY_CLASSES):
    print(f"  {c:10s}: {dist.get(i, 0):>5}")

# Verify a batch
x, y = next(iter(train_loader))
print(f"\nBatch shape: {x.shape}, Labels shape: {y.shape}")

## Cell 5: EfficientNet-B0 Model

Pretrained ImageNet backbone with custom classifier head.
Freeze first 5/7 feature blocks, fine-tune last 2 + head.

In [ ]:
import torch.nn as nn
import torchvision.models as models

class FacialEmotionNet(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        base = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.features = base.features
        self.avgpool = base.avgpool
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(1280, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

model = FacialEmotionNet(NUM_CLASSES).to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"FacialEmotionNet: {total_params:,} total, {trainable:,} trainable")

# Freeze early layers (blocks 0-4 of features)
for i, block in enumerate(model.features):
    if i < 5:
        for param in block.parameters():
            param.requires_grad = False

trainable_after = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"After freezing blocks 0-4: {trainable_after:,} trainable")

# Verify
dummy = torch.randn(2, 3, 224, 224).to(device)
out = model(dummy)
print(f"Output shape: {out.shape}")

## Cell 6: Training Loop

AdamW with differential LR (backbone vs head), cosine schedule, FP16.
~2-3 hours on T4.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
from torch.cuda.amp import autocast, GradScaler

NUM_EPOCHS = 15
PATIENCE = 5

# Class weights for imbalanced data
train_counts = Counter(train_labels)
n_total = len(train_labels)
class_weights = torch.tensor(
    [n_total / (NUM_CLASSES * train_counts.get(i, 1)) for i in range(NUM_CLASSES)],
    dtype=torch.float32
).to(device)
print("Class weights:")
for i, c in enumerate(DISPLAY_CLASSES):
    print(f"  {c}: {class_weights[i]:.3f}")

criterion = nn.CrossEntropyLoss(weight=class_weights)

# Differential learning rates: lower for backbone, higher for head
backbone_params = [p for name, p in model.named_parameters()
                   if 'classifier' not in name and p.requires_grad]
head_params = [p for name, p in model.named_parameters()
               if 'classifier' in name and p.requires_grad]

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': 1e-5},
    {'params': head_params, 'lr': 5e-4},
], weight_decay=1e-2)

total_steps = len(train_loader) * NUM_EPOCHS
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)

# FP16 scaler
scaler = GradScaler()
use_amp = torch.cuda.is_available()

best_val_acc = 0.0
patience_counter = 0

print(f"\n{'='*60}")
print(f"Training: {NUM_EPOCHS} epochs, {len(train_loader)} batches/epoch")
print(f"AMP (FP16): {use_amp}")
print(f"{'='*60}\n")

train_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    correct = 0
    total = 0
    epoch_start = time.time()

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()

        if use_amp:
            with autocast():
                logits = model(x)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        scheduler.step()
        epoch_loss += loss.item()
        preds = logits.argmax(dim=-1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    train_loss = epoch_loss / len(train_loader)
    train_acc = correct / total

    # Validation
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            if use_amp:
                with autocast():
                    logits = model(x)
            else:
                logits = model(x)
            preds = logits.argmax(dim=-1)
            val_correct += (preds == y).sum().item()
            val_total += y.size(0)

    val_acc = val_correct / val_total
    epoch_time = time.time() - epoch_start
    lr_bb = optimizer.param_groups[0]['lr']
    lr_hd = optimizer.param_groups[1]['lr']

    marker = ""
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save({
            'model_state_dict': model.state_dict(),
            'num_classes': NUM_CLASSES,
            'classes': DISPLAY_CLASSES,
            'val_acc': best_val_acc,
            'epoch': epoch,
        }, f"{SAVE_DIR}/best_model.pt")
        marker = " << BEST"
    else:
        patience_counter += 1
        marker = f" ({patience_counter}/{PATIENCE})"

    print(f"Ep {epoch:2d}/{NUM_EPOCHS} ({epoch_time:.0f}s) | "
          f"Loss {train_loss:.4f} | Train {train_acc:.4f} | "
          f"Val {val_acc:.4f}{marker}")

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}")
        break

total_time = time.time() - train_start
print(f"\nTraining complete in {total_time/60:.1f} minutes")
print(f"Best Val Accuracy: {best_val_acc:.4f} (target >= 0.60)")

## Cell 7: Final Evaluation on Test Set

In [ ]:
# Load best model
ckpt = torch.load(f"{SAVE_DIR}/best_model.pt", map_location=device, weights_only=True)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Loaded best model from epoch {ckpt['epoch']}")

# Evaluate on test set
all_preds = []
all_true = []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        if use_amp:
            with autocast():
                logits = model(x)
        else:
            logits = model(x)
        preds = logits.argmax(dim=-1).cpu().tolist()
        all_preds.extend(preds)
        all_true.extend(y.tolist())

test_acc = sum(p == t for p, t in zip(all_preds, all_true)) / len(all_true)

print(f"\n{'='*60}")
print(f"FINAL TEST RESULTS")
print(f"{'='*60}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Target: >= 0.60")
print(f"Result: {'PASSED' if test_acc >= 0.60 else 'BELOW TARGET'}")
print(f"\n{classification_report(all_true, all_preds, target_names=DISPLAY_CLASSES, zero_division=0)}")

# Confusion matrix
cm = confusion_matrix(all_true, all_preds)
print("Confusion Matrix:")
header = "          " + "".join(f"{c[:7]:>8s}" for c in DISPLAY_CLASSES)
print(header)
for i, c in enumerate(DISPLAY_CLASSES):
    print(f"{c:10s}" + "".join(f"{cm[i][j]:8d}" for j in range(NUM_CLASSES)))

if test_acc < 0.60:
    print("\nSuggestions:")
    print("  1. Unfreeze more layers (try freezing only blocks 0-3)")
    print("  2. Increase epochs to 25")
    print("  3. Try label smoothing (0.1)")
    print("  4. Add mixup augmentation")

## Cell 8: Save Model & Metadata

In [ ]:
# Save training metadata
meta = {
    'test_acc': float(test_acc),
    'val_acc': float(best_val_acc),
    'best_epoch': int(ckpt['epoch']),
    'num_classes': NUM_CLASSES,
    'classes': DISPLAY_CLASSES,
    'train_samples': len(train_dataset),
    'val_samples': len(val_dataset),
    'test_samples': len(test_dataset),
    'model_params': sum(p.numel() for p in model.parameters()),
    'backbone': 'efficientnet_b0',
    'input_size': 224,
}
with open(f"{SAVE_DIR}/training_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print(f"Saved to {SAVE_DIR}/")
!ls -lh {SAVE_DIR}/

print(f"\n{'='*60}")
print(f"DONE! Download best_model.pt from Google Drive to your Mac:")
print(f"  mkdir -p ~/Desktop/Claude-assistant/models/facial_emotion/")
print(f"  Place best_model.pt there")
print(f"{'='*60}")

## (Optional) Visualize Predictions

In [ ]:
import matplotlib.pyplot as plt
from torchvision.utils import make_grid

# Get a batch of test images
dataiter = iter(test_loader)
images, labels = next(dataiter)

# Predict
model.eval()
with torch.no_grad():
    outputs = model(images.to(device))
    preds = outputs.argmax(dim=-1).cpu()

# Show 16 images
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for i, ax in enumerate(axes.flat):
    if i >= 16:
        break
    img = images[i].cpu()
    # Denormalize
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img = img * std + mean
    img = img.clamp(0, 1).permute(1, 2, 0).numpy()

    true_label = DISPLAY_CLASSES[labels[i]]
    pred_label = DISPLAY_CLASSES[preds[i]]
    color = 'green' if labels[i] == preds[i] else 'red'

    ax.imshow(img)
    ax.set_title(f"T:{true_label}\nP:{pred_label}", color=color, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/predictions_sample.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved sample predictions to {SAVE_DIR}/predictions_sample.png")